# 各ピークの有意度計算スクリプト

全てのピークの有意度を計算し、最小・最大を出す。


In [84]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.table import Table, vstack
from astropy.modeling.powerlaws import LogParabola1D
from astropy.modeling import models, fitting
from astropy import units as u
from scipy.optimize import curve_fit
from scipy.stats import chi2, norm


## ピーク検出・有意度計算

In [85]:
################################################
### # Peak significance calculation - not used
### This must be used with the listing of all the data points
### of consecutive positive deviations, 
### which can be more than three
################################################
def calc_peak_significance(ri):
#     '''
#     fit (array) values for the fit
#     x,y,yerr (arrays) data
#     N total number of points
#     n_free number of parameters we are fitting
#     '''
#     return 1.0/(N-n_free)*sum(((fit - y)/yerr)**2)
# import numpy as np
# from scipy.stats import chi2, norm
# # only those residuals that form the (at least) 3 consecutive positive deviations
# ri = np.array([1, 2, 3.])

# # Chi-square of those residuals
  chi2_value = np.sum(ri**2)

# # p-value for chi-square with m degrees of freedom - m = len(ri) - this is the probability of obtaining a test statistic at least as high as chi2_value
  p_val = 1 - chi2.cdf(chi2_value, df=len(ri))

# # p-value to Gaussian-equivalent one-sided significance
  sigma_equiv = norm.ppf(1 - p_val)

  return p_val, sigma_equiv


###################
# Peak detection
###################
def get_consecutive_bins(s,nconsecutive = 3 ):
  peakinitbin = -1
  for i in range(0,len(s)-nconsecutive+1):
    # print(i)
    if np.all(s[i:i+nconsecutive] > 1.0):
      #   print(i)
      #   print(s[i:i+nconsecutive])
      #   print('---')
      if peakinitbin < 0:
        peakinitbin = i
        # print('Found peak at bin#:',peakinitbin)
        # print(s[peakinitbin:i+nconsecutive])
      if i + nconsecutive == len(s):
        # print('Last bin reached', i, 'for the length ',len(s))
        # print('End of peak at bin #:',i + nconsecutive -1, 'length ', i + nconsecutive - peakinitbin)
        # print(s[peakinitbin:i+nconsecutive])
        # print(peakinitbin, i+ nconsecutive - peakinitbin)
        return peakinitbin, i+ nconsecutive - peakinitbin, calc_peak_significance(s[peakinitbin:i+nconsecutive])
    else:
      if peakinitbin >= 0:
        # print('End of peak at bin#:',i + nconsecutive -2, 'length ', i -1 + nconsecutive - peakinitbin)
        # print(s[peakinitbin:i+nconsecutive ])
        # print(peakinitbin, i+ nconsecutive -1 - peakinitbin)
        return peakinitbin, i+ nconsecutive -1 - peakinitbin, calc_peak_significance(s[peakinitbin:i+nconsecutive-1])
        # if peakbininit[0] > 0:
        #   print('Found longer peak at bin:',peakbininit[0] , 'in total length of ', len(s))
        # else :
        #   print('Longest peak at bin:',i ,'length ', nconsecutive, 'in total length of ', len(s))
  return -1, 0


## 主体関数

In [86]:

nbinsmin=9

###################
# Utility functions
###################
def gaussian(x, amplitude, mu, sigma):
    # return (1.0 / (sigma * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mu) / sigma) ** 2) : Normal distribution
    return amplitude * np.exp(-0.5 * ((x - mu) / sigma) ** 2)

def calc_reduced_chi_square(fit, x, y, yerr, N, n_free):
    '''
    fit (array) values for the fit
    x,y,yerr (arrays) data
    N total number of points
    n_free number of parameters we are fitting
    '''
    return 1.0/(N-n_free)*sum(((fit - y)/yerr)**2)

###################
# Main function
###################
def eval_spectra(filepath,suffix='',sourcename=''): #'data/3C454.3_allsed_14d_min11.ecsv'
  nconsecutive = 3
  sourcenameheader = ''
  if suffix != '':
    sourcenameheader = suffix + '_'

  if sourcename == '':
    sourcename = filepath.split('_')[0]
    sourcename = sourcename.replace('data/', '')
  # logpar_init = LogParabola1D(amplitude=1,x_0=10,alpha=1,beta=1)
  logpar_init = LogParabola1D(
    amplitude = 0.000016494113774149846,
    x_0 = 9.482855296727278,
    alpha = -0.5677548858840729,
    beta=0)
    # beta = 0.1071613245072173)

  ### initialize a linear fitter ###
  # fit = fitting.TRFLSQFitter()
  fit = fitting.DogBoxLSQFitter()

  ### Read the data ###
  t0 = Table.read(filepath)
  nonzero_mask = (t0['e2dnde'] > t0['e2dnde_err'])
  t = t0[nonzero_mask]

  ### MJDごとに処理 ###
  obsdates=np.unique(t['tstart'].data).tolist()
  n_detected_peaks = 0
  array_nbins = []
  array_chisq = []
  t_peakstat = Table(names = ['length','total length','p-value','sigma'], dtype = ['i2','i2','f4' ,'f4'])
  for idx, obsdate in enumerate(obsdates):
    # print(idx, ': obsdate',obsdate)
    mask = (t['tstart']==obsdate)
    x = t[mask]['e_ref']
    y = t[mask]['e2dnde']
    yerr = t[mask]['e2dnde_err']
    nbins = len(x)
    if nbins < nbinsmin:
        print('nbins < nbinsmin:',nbins)
        continue
    array_nbins.append(nbins)
    
    ### fit the data with the fitter ###
    # logpar_init.amplitude.value=x[1]*1.0e-8
    fitted_line = fit(logpar_init, x,y,weights=1.0/yerr, maxiter=200)
    residuals = (y-fitted_line(x))/yerr
    # print(fit.fit_info)
    # residuals = (fitted_line(x)-y)/yerr # negative:dip finding
    reduced_chi_squared = calc_reduced_chi_square(fitted_line(x), x, y, yerr, len(x), 4)
    array_chisq.append(reduced_chi_squared)

    peakbininit = get_consecutive_bins(residuals,nconsecutive)
    if peakbininit[0] > 0:
      print('Found peak at bin:',peakbininit[0], 'length ', peakbininit[1], 'in total length of ', len(residuals), 'p-value, sigma:', peakbininit[2])
      t_peakstat.add_row([peakbininit[1], len(residuals), peakbininit[2][0], peakbininit[2][1]])
  print(t_peakstat)
  return t_peakstat

## 実行テスト

In [87]:
# eval_spectra('data/3C454.3_allsed_14d_min11.ecsv')  # 2pos, 0neg/264
# eval_spectra('data/3C454.3_allsed_1d_min11.ecsv')  # 8pos
# eval_spectra('data/BLLac_allsed_14d_min11.ecsv')
# eval_spectra('data/BLLac_allsed_1d_min11.ecsv') 
# eval_spectra('data/3C279_allsed_14d_min11.ecsv') 
# eval_spectra('data/3C279_allsed_1d_min11.ecsv') 
# eval_spectra('data/CTA102_allsed_14d_min11.ecsv') 
eval_spectra('data/CTA102_allsed_1d_min11.ecsv') 


Found peak at bin: 1 length  3 in total length of  13 p-value, sigma: (0.2493890378014969, 0.6764136146791156)
nbins < nbinsmin: 8
Found peak at bin: 8 length  3 in total length of  18 p-value, sigma: (0.12845843903181264, 1.133708416386558)
Found peak at bin: 12 length  3 in total length of  15 p-value, sigma: (0.20628094627384264, 0.8193935871307099)
nbins < nbinsmin: 7
nbins < nbinsmin: 7
nbins < nbinsmin: 8
nbins < nbinsmin: 8
nbins < nbinsmin: 8
Found peak at bin: 9 length  4 in total length of  13 p-value, sigma: (0.02618187370159697, 1.940131197748531)
Found peak at bin: 8 length  3 in total length of  11 p-value, sigma: (0.042844845884704275, 1.7185864989367994)
length total length   p-value     sigma  
------ ------------ ----------- ---------
     3           13  0.24938904 0.6764136
     3           18  0.12845844 1.1337084
     3           15  0.20628095 0.8193936
     4           13 0.026181873 1.9401312
     3           11 0.042844847 1.7185864


length,total length,p-value,sigma
int16,int16,float32,float32
3,13,0.24938904,0.6764136
3,18,0.12845844,1.1337084
3,15,0.20628095,0.8193936
4,13,0.026181873,1.9401312
3,11,0.042844847,1.7185864
